In [2]:
import pandas as pd
import sys
import os

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

from src.weak_labeling.model_based.xgboost.xgboost_preprocess import preprocess_for_xgboost_train

if "Desktop" in os.getcwd():
    base_path = "/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification"
else:
    base_path = "/home/sakdag/weakly-supervised-fake-news-classification"

print(base_path)
sys.path.append(base_path)
    
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv
from src.weak_labeling.consts.globals import NLP

/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification


In [3]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

Reading dataset from path: /Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000


In [11]:
from src.weak_labeling.model_based.xgboost.xgboost_train import train_xgb

In [13]:
import datetime

train_xgb(injected_df, save_date=datetime.date(2023, 2, 18))

Use saved XGB model: False


/Users/sakdag/miniconda3/envs/weakly-supervised-fake-news-classification/lib/python3.8/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


The accuracy of XGBOOST is: 0.70
The roc_auc score of XGBOOST is: 0.70
              precision    recall  f1-score   support

           0       0.67      0.77      0.72      5008
           1       0.73      0.62      0.67      4992

    accuracy                           0.70     10000
   macro avg       0.70      0.70      0.69     10000
weighted avg       0.70      0.70      0.69     10000



In [4]:
roberta_train, roberta_test, y_test, y_train = preprocess_for_xgboost_train(injected_df)

# warnings.filterwarnings(action='ignore', category=UserWarning)
parameters = {
    "max_depth": range(2, 10, 1),
    "n_estimators": range(60, 1000, 188),
    "learning_rate": [0.15, 0.1, 0.01, 0.05],
}
xgb = XGBClassifier(
    eval_metric="auc",
    use_label_encoder=False,
    objective="binary:logistic",
    seed=42,
)
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=parameters,
    scoring="roc_auc",
    n_jobs=10,
    cv=10,
    verbose=1,
)
results = grid_search.fit(roberta_train, y_train)
print(results)

print(grid_search.best_estimator_)

# xgb.fit(roberta_train, y_train)
# pickle.dump(xgb, open(xgb_file_name, "wb"))

# xgb_pred = xgb.predict(roberta_test)
# xgb_score = accuracy_score(y_test, xgb_pred)
# xgb_roc = roc_auc_score(y_test, xgb_pred)
# print("The accuracy of XGBOOST is: %0.2f" % xgb_score)
# print("The roc_auc score of XGBOOST is: %0.2f" % xgb_roc)
# cr = classification_report(y_test, xgb_pred)
# print(cr)

Fitting 10 folds for each of 2 candidates, totalling 20 fits


/Users/sakdag/miniconda3/envs/weakly-supervised-fake-news-classification/lib/python3.8/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
/Users/sakdag/miniconda3/envs/weakly-supervised-fake-news-classification/lib/python3.8/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
/Users/sakdag/miniconda3/envs/weakly-supervised-fake-news-classification/lib/python3.8/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


GridSearchCV(cv=10,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='auc', feature_types=None,
                                     gamma=None, gpu_id=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=Non...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=Non

{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 16}